# Chapter 01: Driving Perception Gym & Class Imbalance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/01_driving_perception_gym.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *Why does standard Cross-Entropy ignore pedestrians in driving datasets, and how does Focal Loss mathematically fix it?*

---

## 1. 🚨 The Real-World Dilemma
In driving datasets, 99.1% of pixels belong to empty asphalt, and <0.1% to pedestrians. Standard Cross-Entropy gradients are overwhelmed 4,000-to-1 by easy road pixels. Focal Loss dynamically damps easy examples by $(1 - p_t)^\gamma$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch loaded:", torch.__version__)

## 2. 🛠️ The Karpathy Build: Multi-Class Focal Loss
$$\mathcal{L}_{\text{Focal}}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        modulating = (1.0 - p_t) ** self.gamma
        focal_loss = modulating * ce_loss
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        return focal_loss.mean()

# Compare CE vs Focal Loss suppression
p_vals = torch.linspace(0.01, 0.99, 100)
ce = -torch.log(p_vals)
focal_2 = ((1.0 - p_vals) ** 2.0) * ce

plt.figure(figsize=(8, 3.5))
plt.plot(p_vals.numpy(), ce.numpy(), label="Cross-Entropy (gamma=0)", color="#f85149", lw=2)
plt.plot(p_vals.numpy(), focal_2.numpy(), label="Focal Loss (gamma=2)", color="#58a6ff", lw=2)
plt.title("Loss Damping on Easy Examples (High pt)")
plt.xlabel("Target Probability pt")
plt.ylabel("Loss Magnitude")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **High overall accuracy, 0% recall on rare objects** | Majority class dominance in Cross-Entropy. | Inspect class-stratified confusion matrix. | Switch to Focal Loss with $\gamma = 2.0$. |
| **Loss explodes when weighting classes** | Extreme $\alpha$ multiplier ($>100$) causes massive gradient spikes on outliers. | Check $\max \|\mathbf{g}\|$ per batch. | Clamp class weights $\alpha \le 10.0$ and use gradient clipping. |